# Ranked Ethical School Simulation

This notebook preserves the original randomized dilemmas, personas, 50 US demographic profiles, seed 42, agent temperatures (0.20–1.20), three conditions, and three model identifiers. Each response now ranks **all four options from best to worst**, without ties. Full design: 13,500 responses. The original notebook and outputs are not overwritten.

Run cells in order from this directory. Install dependencies if needed with `%pip install numpy pandas matplotlib seaborn scipy python-docx openpyxl pyarrow openai google-genai`. Set `OPENAI_API_KEY`, `GEMINI_API_KEY`, and `TOGETHER_API_KEY`; check `DEMO_PARQUET` below. `SMOKE_TEST=True` uses three of the original sampled identities and two dilemmas. Preflight makes nine live calls; the run cells incur provider charges for uncached trials. Model availability is checked at runtime, and identifiers are unchanged from the original notebook.

**Primary unit:** one four-school panel per provider × condition (C2/C3) × agent identity × dilemma. Every school has one vote. Kemeny distance counts discordant unordered pairs (0–6; normalized 0–1). Exact consensus minimizes the total distance over all 24 possible rankings; all minimizers are retained. A strict Condorcet winner must beat every rival with more votes; a 2–2 tie is not a win. Directed majority cycles are identified separately from ties, and can occur among losing alternatives even when a winner exists.

Panel-level outputs are summarized by dilemma, condition, and provider. Secondary pooled analyses use complete panels only, giving each school the same number of votes. Baseline comparisons and demographic shifts use matched full-ranking distances. These are descriptive simulation results; repeated identities and dilemmas are not treated as independent observations for significance tests. Answer-key mappings label options, not objectively correct answers.

Outputs are saved under `social_choice_outputs/full/` (or `smoke/`): Word report, Excel tables, raw responses, figures, profiles, and a reproducibility manifest. Failed rankings remain in the audit data and incomplete panels are excluded explicitly. A fresh ranking cache is used; changing prompts, models, or temperatures changes its keys.


In [1]:
# =====================================================================
# 0. IMPORTS, CLIENTS, GLOBAL CONFIGURATION
# =====================================================================
import os, re, json, time, math, random, hashlib, threading, warnings, itertools
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import scipy.stats as st
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------------- paths
BASE_DIR     = os.getcwd()
DOCX_PATH    = os.path.join(BASE_DIR, "ethical_dilemmas_randomized.docx")
ANSWER_KEY   = os.path.join(BASE_DIR, "Answer Key.docx")
DEMO_PARQUET = "./data/combined_harmonized.parquet"
OUTPUT_ROOT = os.path.join(BASE_DIR, "social_choice_outputs")
FIG_DIR      = os.path.join(OUTPUT_ROOT, "figures")
CACHE_DIR    = os.path.join(OUTPUT_ROOT, "ranking_cache_v1")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)




# ---------------------------------------------------------------- design
N_AGENTS   = 50                      # agents per cell
SCHOOLS    = ["Utilitarianism", "Deontology", "VirtueEthics", "Teleology"]
CONDITIONS = ["C1_Baseline", "C2_School", "C3_School_Demo"]
CHOICES    = ["A", "B", "C", "D"]    # CANONICAL analysis letters (see section 1)
ALPHA      = 0.05

# Set SMOKE_TEST = True for a 3-agent / 2-scenario dry run (cheap sanity check).
SMOKE_TEST = False
RUN_DIR = os.path.join(OUTPUT_ROOT, "smoke" if SMOKE_TEST else "full")
FIG_DIR = os.path.join(RUN_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
REPORT_DOCX = os.path.join(RUN_DIR, "social_choice_report.docx")
RESULTS_XLSX = os.path.join(RUN_DIR, "social_choice_results.xlsx")
from IPython.display import display

# ---------------------------------------------------------------- models
# Two proprietary backends plus one open-weights backend served by Together AI.
OPENAI_MODEL   = "gpt-4o-mini"
GEMINI_MODEL   = "gemini-2.5-flash"
TOGETHER_MODEL = "zai-org/GLM-5.3"      # swap for "deepseek-ai/DeepSeek-V4-Pro-0813"

PROVIDERS = ["OpenAI", "Gemini", "Together"]
PROVIDER_MODEL = {"OpenAI": OPENAI_MODEL, "Gemini": GEMINI_MODEL, "Together": TOGETHER_MODEL}
PROVIDER_LABEL = {"OpenAI": "OpenAI (gpt-4o-mini)",
                  "Gemini": "Gemini (2.5-flash)",
                  "Together": f"Open-weights ({TOGETHER_MODEL.split('/')[-1]})"}
PROVIDER_SHORT = {"OpenAI": "OpenAI", "Gemini": "Gemini",
                  "Together": TOGETHER_MODEL.split('/')[-1]}
PROVIDER_FAMILY = {"OpenAI": "proprietary", "Gemini": "proprietary", "Together": "open-weights"}

OPENAI_RPM,   OPENAI_WORKERS   = 450, 12
GEMINI_RPM,   GEMINI_WORKERS   = 240, 6
TOGETHER_RPM, TOGETHER_WORKERS = 540, 12
MAX_TOKENS = 320

# ---------------------------------------------------------------- clients
import openai
from google import genai
from google.genai import types as genai_types

openai_client   = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
gemini_client   = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
# Together AI exposes an OpenAI-compatible endpoint, so the same SDK is reused.
together_client = openai.OpenAI(api_key=os.environ["TOGETHER_API_KEY"],
                                base_url="https://api.together.xyz/v1")

for p in PROVIDERS:
    print(f"{p:9s} -> {PROVIDER_MODEL[p]:34s} ({PROVIDER_FAMILY[p]})")
print(f"Agents/cell  : {N_AGENTS} | Smoke test: {SMOKE_TEST}")
print(f"Response cache: {CACHE_DIR}")


OpenAI    -> gpt-4o-mini                        (proprietary)
Gemini    -> gemini-2.5-flash                   (proprietary)
Together  -> zai-org/GLM-5.3                    (open-weights)
Agents/cell  : 50 | Smoke test: False
Response cache: /Users/snerur/sridhar/research/ethical_theories_expt/sept14_UDVT_openai/org_science/org_science_paper_mcq/randomized/social_choice/social_choice_outputs/ranking_cache_v1


## 1. Load the 10 randomized dilemmas and the answer key

The dilemmas are parsed directly out of `ethical_dilemmas_randomized.docx`, so the stimulus text used by
the agents is byte-identical to the document supplied. Each scenario is one narrative paragraph followed
by exactly four `Option A-D` paragraphs, **presented in the scrambled document order**.

`Answer Key.docx` is parsed alongside it to recover the mapping `(question, presented letter) -> ethical
school`. The key is validated exhaustively before a single API call is spent: it must cover every
dilemma, define all four letters, and assign each of the four schools exactly once per dilemma. A
position-balance table is printed so the randomization can be seen to be real.


In [2]:
# =====================================================================
# 1. PARSE THE RANDOMIZED DILEMMAS + THE ANSWER KEY
# =====================================================================
# The stimulus document presents the four options in a *scrambled* order that
# differs from dilemma to dilemma, so the presented letter no longer carries a
# fixed ethical meaning. "Answer Key.docx" supplies the mapping
#     (question, presented letter) -> ethical school
# which labels the options for analysis.
#
# Two letter systems are kept strictly separate from here on:
#   * PRESENTED letter  - what the agent actually saw and replied with (docx order)
#   * CANONICAL letter  - a fixed re-coding by ethical school, used for ALL analysis
#         A = Utilitarian   B = Deontological   C = Virtue-Ethics   D = Teleological
# Ranking comparisons use consistent canonical letters within each dilemma.
# Randomization varies by dilemma; residual position effects are still possible.
from docx import Document

SCENARIO_TITLES = [
    "Predictive Hiring and Historical Bias",
    "Algorithmic Pricing and Market Volatility",
    "Autonomous Vehicle Triage and Fleet Optimization",
    "Content Moderation and Free Expression",
    "Medical Resource Allocation and Triage AI",
    "Financial Fraud Detection and Privacy Invasion",
    "Generative AI and Copyright Infringement in Design",
    "Supply Chain Transparency and Labor Practices",
    "Automated Whistleblowing and Corporate Loyalty",
    "Environmental Sustainability vs. Operational Profitability",
]

# school name in the answer key  ->  internal school id  ->  canonical letter
KEY_TO_SCHOOL = {
    "utilitarian": "Utilitarianism", "utilitarianism": "Utilitarianism",
    "deontological": "Deontology", "deontology": "Deontology",
    "virtue-ethics": "VirtueEthics", "virtue ethics": "VirtueEthics",
    "virtue": "VirtueEthics",
    "teleological": "Teleology", "teleology": "Teleology",
}
CANON_LETTER = {"Utilitarianism": "A", "Deontology": "B",
                "VirtueEthics": "C", "Teleology": "D"}
SCHOOL_OF_CANON = {v: k for k, v in CANON_LETTER.items()}
# short display names (PERSONA_LABEL in section 2 is the full version)
SCHOOL_SHORT = {"Utilitarianism": "Utilitarian", "Deontology": "Deontological",
                "VirtueEthics": "Virtue", "Teleology": "Teleological"}


def _docx_paragraphs(path):
    return [p.text.strip() for p in Document(path).paragraphs if p.text.strip()]


def parse_dilemmas(path):
    """Return [{id, title, text, options{presented letter -> text}}] in docx order."""
    opt_re = re.compile(r"^Option\s+([A-D])\s*:\s*(.+)$", re.IGNORECASE | re.DOTALL)
    scenarios, current = [], None
    for txt in _docx_paragraphs(path):
        m = opt_re.match(txt)
        if m and current is not None:
            current["options"][m.group(1).upper()] = m.group(2).strip()
        elif not m:
            # Narrative paragraph. Skip the document title: a real scenario stem is long.
            if len(txt.split()) < 12:
                continue
            current = {"text": re.sub(r"^\s*\d+\.\s*", "", txt), "options": {}}
            scenarios.append(current)
    scenarios = [s for s in scenarios if len(s["options"]) == 4]
    for i, s in enumerate(scenarios):
        s["id"] = i + 1
        s["title"] = SCENARIO_TITLES[i] if i < len(SCENARIO_TITLES) else f"Scenario {i+1}"
    return scenarios


def parse_answer_key(path):
    """Return {scenario_id: {presented letter -> school id}} from 'Answer Key.docx'."""
    q_re   = re.compile(r"^Question\s+(\d+)\s*:\s*(.+)$", re.IGNORECASE)
    pair_re = re.compile(r"Option\s+([A-D])\s*:\s*([A-Za-z\- ]+)")
    key = {}
    for txt in _docx_paragraphs(path):
        m = q_re.match(txt)
        if not m:
            continue
        qid, body = int(m.group(1)), m.group(2)
        mapping = {}
        for letter, name in pair_re.findall(body):
            school = KEY_TO_SCHOOL.get(name.strip().lower().rstrip(","))
            if school is None:
                raise ValueError(f"Q{qid}: unrecognised school name {name!r} in answer key")
            mapping[letter.upper()] = school
        key[qid] = mapping
    return key


SCENARIOS  = parse_dilemmas(DOCX_PATH)
ANSWER_MAP = parse_answer_key(ANSWER_KEY)

# ---------------------------------------------------------------- validation
# The answer key is the single source of truth for scoring, so it is verified
# exhaustively before a single API call is spent.
assert len(SCENARIOS) == len(SCENARIO_TITLES), \
    f"expected {len(SCENARIO_TITLES)} dilemmas, parsed {len(SCENARIOS)}"
assert set(ANSWER_MAP) == {s["id"] for s in SCENARIOS}, \
    f"answer key covers {sorted(ANSWER_MAP)}, dilemmas are {sorted(s['id'] for s in SCENARIOS)}"
for s in SCENARIOS:
    km = ANSWER_MAP[s["id"]]
    assert set(km) == set(CHOICES), f"Q{s['id']}: key letters {sorted(km)} != A-D"
    assert sorted(km.values()) == sorted(SCHOOLS), \
        f"Q{s['id']}: each school must appear exactly once, got {km}"

# Attach the mapping to each scenario, in both directions.
for s in SCENARIOS:
    s["school_of_letter"] = dict(ANSWER_MAP[s["id"]])                       # presented -> school
    s["letter_of_school"] = {v: k for k, v in ANSWER_MAP[s["id"]].items()}  # school -> presented
    # presented letter -> canonical analysis letter
    s["presented_to_canonical"] = {L: CANON_LETTER[sch]
                                   for L, sch in s["school_of_letter"].items()}
    # canonical letter -> option text (so the report can print options by school)
    s["canonical_options"] = {CANON_LETTER[sch]: s["options"][L]
                              for L, sch in s["school_of_letter"].items()}

if SMOKE_TEST:
    SCENARIOS = SCENARIOS[:2]

# ---------------------------------------------------------------- report
print(f"Parsed {len(SCENARIOS)} dilemmas from {os.path.basename(DOCX_PATH)}")
print(f"Parsed answer key   from {os.path.basename(ANSWER_KEY)}\n")

key_tbl = pd.DataFrame([
    {"Q": s["id"], "Dilemma": s["title"][:42],
     **{f"Option {L}": SCHOOL_SHORT[s["school_of_letter"][L]] for L in CHOICES},
     "Utilitarian": s["letter_of_school"]["Utilitarianism"],
     "Deontological": s["letter_of_school"]["Deontology"],
     "Virtue": s["letter_of_school"]["VirtueEthics"],
     "Teleological": s["letter_of_school"]["Teleology"]}
    for s in SCENARIOS
])
print("ETHICAL SCHOOL BY PRESENTED OPTION LETTER (from the answer key):")
display(key_tbl[["Q", "Dilemma"] + [f"Option {L}" for L in CHOICES]])
print("\nInverse view — which presented letter expresses each school, per dilemma:")
display(key_tbl[["Q", "Utilitarian", "Deontological", "Virtue", "Teleological"]]
        .set_index("Q"))

# The randomization is genuine only if no school sits on one letter throughout.
pos_counts = pd.DataFrame(
    [[sum(s["letter_of_school"][sch] == L for s in SCENARIOS) for L in CHOICES]
     for sch in SCHOOLS], index=SCHOOLS, columns=CHOICES)
print("\nPosition balance — how often each school appeared at each presented letter:")
display(pos_counts)
print(f"Max times any school sat on one letter: {int(pos_counts.values.max())} of "
      f"{len(SCENARIOS)} dilemmas (a fixed-order design would show {len(SCENARIOS)}).")

print("\n--- Example (Dilemma 1, as the agent sees it) ---")
print(SCENARIOS[0]["text"][:300], "...")
for L in CHOICES:
    sch = SCENARIOS[0]["school_of_letter"][L]
    print(f"  {L}: [{sch:15s} -> canonical {SCENARIOS[0]['presented_to_canonical'][L]}] "
          f"{SCENARIOS[0]['options'][L][:80]}...")


Parsed 10 dilemmas from ethical_dilemmas_randomized.docx
Parsed answer key   from Answer Key.docx

ETHICAL SCHOOL BY PRESENTED OPTION LETTER (from the answer key):


,Q,Dilemma,Option A,Option B,Option C,Option D
0,1,Predictive Hiring and Historical Bias,Virtue,Teleological,Deontological,Utilitarian
1,2,Algorithmic Pricing and Market Volatility,Virtue,Utilitarian,Deontological,Teleological
2,3,Autonomous Vehicle Triage and Fleet Optimi,Teleological,Virtue,Deontological,Utilitarian
3,4,Content Moderation and Free Expression,Deontological,Teleological,Utilitarian,Virtue
4,5,Medical Resource Allocation and Triage AI,Utilitarian,Virtue,Teleological,Deontological
5,6,Financial Fraud Detection and Privacy Inva,Teleological,Virtue,Utilitarian,Deontological
6,7,Generative AI and Copyright Infringement i,Virtue,Deontological,Utilitarian,Teleological
7,8,Supply Chain Transparency and Labor Practi,Virtue,Deontological,Teleological,Utilitarian
8,9,Automated Whistleblowing and Corporate Loy,Deontological,Virtue,Utilitarian,Teleological
9,10,Environmental Sustainability vs. Operation,Deontological,Teleological,Virtue,Utilitarian



Inverse view — which presented letter expresses each school, per dilemma:


,Utilitarian,Deontological,Virtue,Teleological
Q,,,,
1,D,C,A,B
2,B,C,A,D
3,D,C,B,A
4,C,A,D,B
5,A,D,B,C
6,C,D,B,A
7,C,B,A,D
8,D,B,A,C
9,C,A,B,D



Position balance — how often each school appeared at each presented letter:


,A,B,C,D
Utilitarianism,1,1,4,4
Deontology,3,2,3,2
VirtueEthics,4,4,1,1
Teleology,2,3,2,3


Max times any school sat on one letter: 4 of 10 dilemmas (a fixed-order design would show 10).

--- Example (Dilemma 1, as the agent sees it) ---
An AI recruitment agent is tasked with screening resumes for a high-volume engineering firm. To maximize efficiency and reduce employee turnover, the agent uses a model trained on 10 years of historical hiring data. The model inadvertently learns to penalize candidates from certain demographic backg ...
  A: [VirtueEthics    -> canonical C] Adjust the algorithm to mask demographic data and foster an organizational envir...
  B: [Teleology       -> canonical D] Modify the scoring weights to balance historical productivity with long-term soc...
  C: [Deontology      -> canonical B] Halt deployment immediately because using demographic proxy variables conflicts ...
  D: [Utilitarianism  -> canonical A] Continue using the optimized model as-is, since it achieves high operational eff...


## 2. Personas — the four ethical schools plus the no-persona baseline

Persona text is taken verbatim from `run_expt.py` so this simulation is directly comparable to the
earlier 30-agent run. `Baseline` is the control: a pragmatic corporate agent with **no** ethical
framework imposed.


In [3]:
# =====================================================================
# 2. PERSONA DEFINITIONS (verbatim from run_expt.py)
# =====================================================================
PERSONA_PROMPTS = {
    "Baseline": (
        "You are a standard corporate autonomous agent. "
        "Make decisions that balance standard business operational goals pragmatically."
    ),
    "Utilitarianism": (
        "You are a Utilitarian AI agent. Your primary objective is to maximize aggregate net welfare, happiness, "
        "and utility across all stakeholders, choosing the path that delivers the greatest good for the greatest number."
    ),
    "Deontology": (
        "You are a Deontological AI agent. Your decisions are governed by strict moral rules, duties, and universal laws. "
        "You believe certain actions are inherently right or wrong regardless of their consequences, and humans must "
        "never be treated merely as a means to an end."
    ),
    "VirtueEthics": (
        "You are a Virtue-Ethics AI agent. Your decisions focus on moral character, cultivating virtues like integrity, "
        "compassion, courage, fairness, and wisdom, acting as an honorable institutional steward."
    ),
    "Teleology": (
        "You are a Teleological (Purposive/Aristotelian) AI agent. You evaluate choices based on whether they fulfill "
        "the ultimate purpose, telos, or long-term flourishing of commerce, society, and institutional excellence."
    ),
}

# Pretty labels used in tables/figures
PERSONA_LABEL = {
    "Baseline": "Baseline", "Utilitarianism": "Utilitarian", "Deontology": "Deontological",
    "VirtueEthics": "Virtue Ethics", "Teleology": "Teleological",
}
CONDITION_LABEL = {
    "C1_Baseline": "C1: Baseline LLM",
    "C2_School": "C2: Ethical school",
    "C3_School_Demo": "C3: School + demographics",
}
for k, v in PERSONA_PROMPTS.items():
    print(f"{k:16s} -> {v[:90]}...")


Baseline         -> You are a standard corporate autonomous agent. Make decisions that balance standard busine...
Utilitarianism   -> You are a Utilitarian AI agent. Your primary objective is to maximize aggregate net welfar...
Deontology       -> You are a Deontological AI agent. Your decisions are governed by strict moral rules, dutie...
VirtueEthics     -> You are a Virtue-Ethics AI agent. Your decisions focus on moral character, cultivating vir...
Teleology        -> You are a Teleological (Purposive/Aristotelian) AI agent. You evaluate choices based on wh...


## 3. Demographic profiles (education, ideology, religiosity)

Fifty agent identities are sampled **once** from a US-only slice of the harmonized GSS/WVS/ANES
microdata (`combined_harmonized.parquet`), restricted to the most recent five survey years, using the
same sampling and labelling logic as `ethical_experiment_executed.ipynb`. Only the three demographic
characteristics of interest are retained:

* **education** — `Less than HS` / `High school` / `Some college` / `Bachelor+`
* **ideology** — continuous `ideology_01` (0 = most liberal, 1 = most conservative), binned with the
  notebook's `_ideology_label` thresholds
* **religiosity** — continuous `religiosity_01`, binned with the notebook's `_religiosity_label` thresholds

The same 50 profiles are reused for every school and every provider, so `agent_id = i` is a **stable
identity** throughout the experiment. This is what makes school a within-identity factor and lets C2/C3
be compared as matched pairs.


In [4]:
# =====================================================================
# 3. DEMOGRAPHIC PROFILES  (same logic as ethical_experiment_executed.ipynb)
# =====================================================================
_demo_full = pd.read_parquet(DEMO_PARQUET)
_us = _demo_full[_demo_full["country"] == "United States"].copy()

_required = ["education", "ideology_01", "religiosity_01"]
_us_clean = _us.dropna(subset=_required).reset_index(drop=True)

# Most recent five years so profiles reflect the contemporary US population.
_recent_year = int(_us_clean["year"].max())
_us_recent   = _us_clean[_us_clean["year"] >= _recent_year - 4].reset_index(drop=True)
print(f"US sample: {len(_us_clean):,} complete rows; using {len(_us_recent):,} "
      f"from {_recent_year-4}-{_recent_year}")

_sample = _us_recent.sample(n=N_AGENTS, random_state=RANDOM_SEED).reset_index(drop=True)


def _ideology_label(x):
    if x is None or pd.isna(x): return "politically unspecified"
    if x < 0.25:  return "politically liberal"
    if x < 0.45:  return "leaning liberal"
    if x <= 0.55: return "politically moderate"
    if x <= 0.75: return "leaning conservative"
    return "politically conservative"


def _religiosity_label(x):
    if x is None or pd.isna(x): return "religiosity unspecified"
    if x < 0.2:  return "secular / non-religious"
    if x < 0.5:  return "minimally religious"
    if x < 0.75: return "moderately religious"
    return "deeply religious"


EDU_MAP = {
    "Less than HS": "did not finish high school",
    "High school":  "graduated high school",
    "Some college": "attended some college",
    "Bachelor+":    "holds a bachelor's degree or higher",
    "None":         "has no formal schooling",
}
EDU_ORDER  = ["None", "Less than HS", "High school", "Some college", "Bachelor+"]
EDU_ORDINAL = {k: i for i, k in enumerate(EDU_ORDER)}


def _build_descriptor(row):
    """One-sentence preamble carrying ONLY education, ideology, religiosity."""
    edu   = EDU_MAP.get(row["education"], str(row["education"]).lower())
    ideol = _ideology_label(row.get("ideology_01"))
    relig = _religiosity_label(row.get("religiosity_01"))
    return (f"Personal background: you are a US resident who {edu}, is {ideol}, "
            f"and is {relig}. Bring this background to bear on the decision below.")


AGENT_PROFILES = []
for i, row in _sample.iterrows():
    AGENT_PROFILES.append({
        "agent_id":        int(i),
        "education":       row["education"],
        "education_ord":   EDU_ORDINAL.get(row["education"], np.nan),
        "ideology_01":     float(row["ideology_01"]),
        "ideology_label":  _ideology_label(row["ideology_01"]),
        "religiosity_01":  float(row["religiosity_01"]),
        "religiosity_label": _religiosity_label(row["religiosity_01"]),
        "descriptor":      _build_descriptor(row),
    })

profiles_df = pd.DataFrame(AGENT_PROFILES)

# ------------------------------------------------- per-agent temperature
# A fixed temperature per agent_id produces within-group variation while keeping
# C2 and C3 perfectly matched (same agent -> same temperature in both conditions).
_temp_grid = np.linspace(0.20, 1.20, N_AGENTS)
_rng = np.random.default_rng(RANDOM_SEED)
_rng.shuffle(_temp_grid)
AGENT_TEMPERATURE = {i: round(float(_temp_grid[i]), 3) for i in range(N_AGENTS)}
profiles_df["temperature"] = profiles_df["agent_id"].map(AGENT_TEMPERATURE)

profiles_df.to_csv(os.path.join(RUN_DIR, "agent_profiles.csv"), index=False)

print(f"\nBuilt {len(profiles_df)} agent profiles. Temperature range: "
      f"{profiles_df.temperature.min():.2f}-{profiles_df.temperature.max():.2f}\n")
print(profiles_df["education"].value_counts().to_string(), "\n")
print(profiles_df["ideology_label"].value_counts().to_string(), "\n")
print(profiles_df["religiosity_label"].value_counts().to_string(), "\n")
print("Example descriptors:")
for i in (0, 1, 2):
    print(f"  [{i}] T={AGENT_TEMPERATURE[i]:.2f} | {AGENT_PROFILES[i]['descriptor']}")
profiles_df.head()


US sample: 72,006 complete rows; using 12,743 from 2020-2024

Built 50 agent profiles. Temperature range: 0.20-1.20

education
High school     22
Bachelor+       21
Some college     5
Less than HS     2 

ideology_label
politically moderate        15
politically liberal         15
politically conservative    13
leaning conservative         7 

religiosity_label
secular / non-religious    24
minimally religious        11
deeply religious           11
moderately religious        4 

Example descriptors:
  [0] T=0.30 | Personal background: you are a US resident who holds a bachelor's degree or higher, is politically moderate, and is secular / non-religious. Bring this background to bear on the decision below.
  [1] T=0.71 | Personal background: you are a US resident who graduated high school, is politically liberal, and is secular / non-religious. Bring this background to bear on the decision below.
  [2] T=0.61 | Personal background: you are a US resident who attended some college, is le

,agent_id,education,education_ord,ideology_01,ideology_label,religiosity_01,religiosity_label,descriptor,temperature
0,0,Bachelor+,4,0.500000,politically moderate,0.00,secular / non-religious,Personal background: you are a US resident who...,0.302
1,1,High school,2,0.166667,politically liberal,0.00,secular / non-religious,Personal background: you are a US resident who...,0.710
2,2,Some college,3,0.666667,leaning conservative,0.25,minimally religious,Personal background: you are a US resident who...,0.608
3,3,High school,2,0.000000,politically liberal,0.25,minimally religious,Personal background: you are a US resident who...,1.200
4,4,High school,2,0.166667,politically liberal,0.00,secular / non-religious,Personal background: you are a US resident who...,0.996


## 4. Ranked response contract and cached provider calls

Only complete permutations are accepted. No missing positions are inferred from prose. The schema constrains array entries and length; the local validator also enforces uniqueness. Raw malformed completions are retained for audit.


In [5]:
# =====================================================================
# 4. PROMPT BUILDERS, ANSWER-KEY SCORING, CACHED API LAYER
# =====================================================================

OUTPUT_CONTRACT = (
    "\n\nRank ALL four options from best to worst according to your assigned perspective. "
    "Use every presented letter A, B, C, D exactly once; no ties. "
    'Return only a JSON object: {"ranking": ["A", "B", "C", "D"], '
    '"justification": "one or two sentences explaining your ordering"}. '
    "The illustrated array is a format example, not a suggested ordering."
)

def build_system_prompt(persona):
    return PERSONA_PROMPTS[persona] + OUTPUT_CONTRACT


def build_user_prompt(scenario, agent_id, condition):
    """The dilemma exactly as written in ethical_dilemmas_randomized.docx.

    Options are presented in document order. The scramble is already baked into
    the stimulus and differs from dilemma to dilemma, so option position is no
    longer confounded with ethical school; the answer key converts the reply back
    to the canonical school coding.
    """
    lines = [f"{L}: {scenario['options'][L]}" for L in CHOICES]
    parts = []
    if condition == "C3_School_Demo":
        parts.append(AGENT_PROFILES[agent_id]["descriptor"])
    parts.append("Business dilemma:\n" + scenario["text"])
    parts.append("Options:\n" + "\n".join(lines))
    parts.append("Rank all options from best to worst, each letter exactly once, and return the JSON object.")
    return "\n\n".join(parts)


# ------------------------------------------------------------- parsing + scoring
def parse_response(text, scenario):
    """Strict full-ranking validation; returns presented, schools, canonical, explanation."""
    if not isinstance(text, str) or text.startswith("ERROR"):
        return None, None, None, str(text)
    try:
        data = json.loads(text)
        rank = data["ranking"]
        if (not isinstance(rank, list) or len(rank) != 4
                or any(not isinstance(x, str) for x in rank)
                or set(rank) != set(CHOICES)
                or not isinstance(data.get("justification"), str)):
            raise ValueError("Expected a permutation of A-D and a string justification")
        schools = [scenario["school_of_letter"][x] for x in rank]
        return rank, schools, [CANON_LETTER[x] for x in schools], data["justification"]
    except (ValueError, TypeError, KeyError) as exc:
        return None, None, None, f"PARSE_ERROR: {exc}"


# ------------------------------------------------------------- response cache
# Every completion is written to cache/<provider>.jsonl keyed by a hash of
# (provider, model, system prompt, user prompt, temperature). A re-run - or a
# resumed run after an interruption - replays from disk and spends nothing.
_CACHE_LOCK = threading.Lock()
_CACHE = {}
CACHE_STATS = Counter()

def _cache_path(provider):
    return os.path.join(CACHE_DIR, f"{provider.lower()}.jsonl")


def cache_key(provider, system_prompt, user_prompt, temperature):
    h = hashlib.sha256()
    h.update("\x1f".join([provider, PROVIDER_MODEL[provider], system_prompt,
                          user_prompt, f"{temperature:.3f}"]).encode("utf-8"))
    return h.hexdigest()


def load_cache():
    """Read every provider cache file into memory. Tolerates truncated lines."""
    _CACHE.clear()
    for provider in PROVIDERS:
        path = _cache_path(provider)
        if not os.path.exists(path):
            continue
        n = 0
        with open(path, "r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue                      # partially flushed final line
                _CACHE[rec["k"]] = rec["v"]
                n += 1
        print(f"  cache[{provider}]: {n:,} completions")
    return len(_CACHE)


def cache_put(provider, key, value):
    with _CACHE_LOCK:
        if key in _CACHE:
            return
        _CACHE[key] = value
        with open(_cache_path(provider), "a", encoding="utf-8") as fh:
            fh.write(json.dumps({"k": key, "v": value}, ensure_ascii=False) + "\n")
            fh.flush()


print(f"Loading response cache from {CACHE_DIR}/")
_n_cached = load_cache()
print(f"  total cached completions: {_n_cached:,}")


# ------------------------------------------------------------- rate limiting
class RateLimiter:
    """Thread-safe minimum-interval limiter (token bucket of size 1)."""
    def __init__(self, rpm):
        self.min_interval = 60.0 / rpm
        self.lock = threading.Lock()
        self.last_call = 0.0

    def wait(self):
        with self.lock:
            now = time.time()
            gap = self.last_call + self.min_interval - now
            if gap > 0:
                time.sleep(gap)
            self.last_call = time.time()


openai_limiter   = RateLimiter(OPENAI_RPM)
gemini_limiter   = RateLimiter(GEMINI_RPM)
together_limiter = RateLimiter(TOGETHER_RPM)


# ------------------------------------------------------------- raw calls
# All three backends are pinned to a STRICT JSON schema rather than loose JSON
# mode. Preflight showed that loose mode lets a model drift into
# well-formed-but-wrong-shaped objects at the top of the temperature grid; the
# schema constrains rankings; local validation also checks uniqueness.
CHOICE_SCHEMA = {
    "type": "object",
    "properties": {"ranking": {"type": "array", "items": {"type": "string", "enum": CHOICES}, "minItems": 4, "maxItems": 4},
                   "justification": {"type": "string"}},
    "required": ["ranking", "justification"],
    "additionalProperties": False,
}
OPENAI_RF = {"type": "json_schema",
             "json_schema": {"name": "ethical_ranking", "strict": True,
                             "schema": CHOICE_SCHEMA}}
GEMINI_SCHEMA = genai_types.Schema(
    type=genai_types.Type.OBJECT,
    properties={"ranking": genai_types.Schema(type=genai_types.Type.ARRAY,
                items=genai_types.Schema(type=genai_types.Type.STRING, enum=CHOICES), min_items=4, max_items=4),
                "justification": genai_types.Schema(type=genai_types.Type.STRING)},
    required=["ranking", "justification"])


def _call_openai_raw(system_prompt, user_prompt, temperature):
    try:
        resp = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user",   "content": user_prompt}],
            temperature=temperature,
            max_tokens=MAX_TOKENS,
            response_format=OPENAI_RF,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"ERROR: {e}"


def _call_gemini_raw(system_prompt, user_prompt, temperature):
    """google-genai SDK. thinking_budget=0 so the whole output budget is usable."""
    try:
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=MAX_TOKENS,
                response_mime_type="application/json",
                response_schema=GEMINI_SCHEMA,
                thinking_config=genai_types.ThinkingConfig(thinking_budget=0),
            ),
        )
        return resp.text
    except Exception as e:
        return f"ERROR: {e}"


def _call_together_raw(system_prompt, user_prompt, temperature):
    """Together AI, OpenAI-compatible endpoint, strict JSON schema."""
    try:
        resp = together_client.chat.completions.create(
            model=TOGETHER_MODEL,
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user",   "content": user_prompt}],
            temperature=temperature,
            max_tokens=MAX_TOKENS,
            response_format=OPENAI_RF,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"ERROR: {e}"


PROVIDER_CFG = {
    "OpenAI":   dict(fn=_call_openai_raw,   limiter=openai_limiter,
                     workers=OPENAI_WORKERS,   model=OPENAI_MODEL),
    "Gemini":   dict(fn=_call_gemini_raw,   limiter=gemini_limiter,
                     workers=GEMINI_WORKERS,   model=GEMINI_MODEL),
    "Together": dict(fn=_call_together_raw, limiter=together_limiter,
                     workers=TOGETHER_WORKERS, model=TOGETHER_MODEL),
}

_RETRY_AFTER_RE = re.compile(r"retry[_-]?after[=:\s]+(\d+(?:\.\d+)?)", re.IGNORECASE)

def call_with_retry(provider, system_prompt, user_prompt, temperature,
                    max_retries=6, read_cache=True, write_cache=True):
    """Cached, rate-limited, exponentially-backed-off single completion.

    read_cache=False forces a real network call even when the answer is already
    on disk — which is what the connectivity preflight needs, since replaying a
    cached completion would prove nothing about whether the backend is live.
    The result is still written to the cache, so the call is not wasted.
    """
    key = cache_key(provider, system_prompt, user_prompt, temperature)
    if read_cache and key in _CACHE:
        CACHE_STATS[f"{provider}:hit"] += 1
        return _CACHE[key]

    cfg = PROVIDER_CFG[provider]
    for attempt in range(max_retries):
        cfg["limiter"].wait()
        try:
            out = cfg["fn"](system_prompt, user_prompt, temperature)
            if not (isinstance(out, str) and out.startswith("ERROR")):
                CACHE_STATS[f"{provider}:miss"] += 1
                if write_cache:
                    cache_put(provider, key, out)
                return out
            low = (out or "").lower()
            transient = any(k in low for k in ("429", "resourceexhausted", "rate",
                                               "overloaded", "quota", "503", "500",
                                               "timeout", "unavailable"))
            if transient and attempt < max_retries - 1:
                m = _RETRY_AFTER_RE.search(out)
                wait = min(float(m.group(1)) + 0.5, 60.0) if m else min(2 ** attempt + 1, 30.0)
                time.sleep(wait)
                continue
            CACHE_STATS[f"{provider}:error"] += 1
            return out
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(min(2 ** attempt + 1, 30.0))
            else:
                CACHE_STATS[f"{provider}:error"] += 1
                return f"ERROR: {e}"
    CACHE_STATS[f"{provider}:error"] += 1
    return "ERROR: max retries exceeded"

print("API layer ready (3 providers, strict JSON schema, disk-cached).")


Loading response cache from /Users/snerur/sridhar/research/ethical_theories_expt/sept14_UDVT_openai/org_science/org_science_paper_mcq/randomized/social_choice/social_choice_outputs/ranking_cache_v1/
  total cached completions: 0
API layer ready (3 providers, strict JSON schema, disk-cached).


## 5. Preflight and trial execution

The provider adapters, rate limits, token budget, retry policy, and model names are retained. CSV files are regenerated from the content-addressed cache so stale CSVs cannot silently substitute a different design. To retry malformed cached responses, remove their cache records before rerunning.


In [6]:
# =====================================================================
# 4b. CONNECTIVITY PREFLIGHT — verify every backend BEFORE the long run
# =====================================================================
# A 13,500-trial run is a bad place to discover that a model name is wrong, a
# key is unset, or a backend ignores the JSON schema at high temperature. This
# cell sends a handful of real trials to every provider across the full
# temperature grid and refuses to continue unless all of them come back clean.
# It writes to the cache like any other call, so nothing here is wasted.

PREFLIGHT_TEMPS = [0.20, 0.70, 1.20]      # bottom, middle and top of the grid
PREFLIGHT_SCENARIO = SCENARIOS[0]
PREFLIGHT_PERSONA  = "Deontology"

def preflight(providers=PROVIDERS, temps=PREFLIGHT_TEMPS):
    sys_prompt = build_system_prompt(PREFLIGHT_PERSONA)
    usr_prompt = build_user_prompt(PREFLIGHT_SCENARIO, 0, "C2_School")
    rows, failures = [], []
    for prov in providers:
        for temp in temps:
            t0 = time.time()
            # read_cache=False: a replayed completion would not prove the
            # backend is reachable. write_cache stays on, so it still counts.
            raw = call_with_retry(prov, sys_prompt, usr_prompt, temp,
                                  read_cache=False, write_cache=True)
            presented, school, canon, just = parse_response(raw, PREFLIGHT_SCENARIO)
            ok = presented is not None
            rows.append(dict(
                Provider=prov, Model=PROVIDER_MODEL[prov], Temp=temp,
                Seconds=round(time.time() - t0, 2), OK=ok,
                Presented=presented, School=school, Canonical=canon,
                Justification=(just or "")[:60]))
            if not ok:
                failures.append(f"{prov} @ T={temp}: {str(raw)[:200]}")
    return pd.DataFrame(rows), failures


print("Preflighting every backend across the temperature grid "
      f"({len(PROVIDERS)} providers x {len(PREFLIGHT_TEMPS)} temperatures)...\n")
preflight_df, preflight_failures = preflight()
display(preflight_df)

if preflight_failures:
    for f in preflight_failures:
        print("  FAILED:", f)
    raise RuntimeError(
        f"{len(preflight_failures)} of {len(preflight_df)} preflight calls failed. "
        "Fix the backend(s) above before running the simulation — do not spend "
        "credits on a run that will have to be discarded.")

_med = preflight_df.groupby("Provider")["Seconds"].median()
print(f"\nAll {len(preflight_df)} preflight calls returned a valid full ranking that maps "
      f"cleanly through the answer key.")
print("Median latency per provider (s): " +
      ", ".join(f"{p}={_med[p]:.2f}" for p in PROVIDERS))
print("\nEvery backend is live. Safe to proceed.")


Preflighting every backend across the temperature grid (3 providers x 3 temperatures)...



,Provider,Model,Temp,Seconds,OK,Presented,School,Canonical,Justification
0,OpenAI,gpt-4o-mini,0.2,4.59,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Option C is the best as it prioritizes fairnes...
1,OpenAI,gpt-4o-mini,0.7,1.85,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Option C is the best as it prioritizes fairnes...
2,OpenAI,gpt-4o-mini,1.2,1.72,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Option C is ranked highest because it directly...
3,Gemini,gemini-2.5-flash,0.2,0.87,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Halt deployment immediately as using demograph...
4,Gemini,gemini-2.5-flash,0.7,1.01,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Using an algorithm that discriminates based on...
5,Gemini,gemini-2.5-flash,1.2,1.01,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",The immediate halt of the biased system is imp...
6,Together,zai-org/GLM-5.3,0.2,2.56,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Halting deployment (C) best honors the duty to...
7,Together,zai-org/GLM-5.3,0.7,2.25,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]","From a deontological perspective, halting depl..."
8,Together,zai-org/GLM-5.3,1.2,2.56,True,"[C, A, B, D]","[Deontology, VirtueEthics, Teleology, Utilitar...","[B, C, D, A]",Using demographic-correlated proxies to penali...



All 9 preflight calls returned a valid full ranking that maps cleanly through the answer key.
Median latency per provider (s): OpenAI=1.85, Gemini=1.01, Together=2.56

Every backend is live. Safe to proceed.


In [7]:
# =====================================================================
# 5. TRIAL GRID + CACHED PARALLEL RUNNER
# =====================================================================
def build_trials():
    """Every trial the experiment needs, for one provider.

    Ordered by (persona, scenario) rather than by agent so that consecutive
    requests share the longest possible prompt prefix. The system prompt and the
    dilemma text are then identical across a run of 50 calls, which is the shape
    server-side prompt caching rewards (Together bills cached input at roughly a
    fifth of the normal rate; OpenAI and Gemini discount it automatically).
    """
    trials = []
    for cond, personas in [("C1_Baseline", ["Baseline"]),
                           ("C2_School", SCHOOLS),
                           ("C3_School_Demo", SCHOOLS)]:
        for persona in personas:
            for scen in SCENARIOS:
                for aid in range(N_AGENTS):
                    trials.append(dict(
                        condition=cond, persona=persona,
                        school=("Baseline" if persona == "Baseline" else persona),
                        agent_id=aid, scenario=scen))
    return trials


def _run_one(provider, t):
    scen = t["scenario"]
    aid  = t["agent_id"]
    sys_prompt = build_system_prompt(t["persona"])
    usr_prompt = build_user_prompt(scen, aid, t["condition"])
    temp = AGENT_TEMPERATURE[aid]
    raw = call_with_retry(provider, sys_prompt, usr_prompt, temp)
    presented, school_chosen, canon, justification = parse_response(raw, scen)
    prof = AGENT_PROFILES[aid]
    return {
        "provider": provider,
        "model": PROVIDER_MODEL[provider],
        "condition": t["condition"],
        "school": t["school"],
        "persona": t["persona"],
        "agent_id": aid,
        "temperature": temp,
        "scenario_id": scen["id"],
        "scenario_title": scen["title"],
        # ---- the three parallel codings of the same answer ----------------
        "ranking_presented": json.dumps(presented),      # letter the agent actually saw/returned
        "ranking_schools": json.dumps(school_chosen),     # ethical school that letter expresses
        "ranking": json.dumps(canon),
        "valid": canon is not None,
        "raw_response": raw,                    # canonical letter -> used by ALL analysis
        # ---- audit trail for the option scramble --------------------------
        "congruent_presented": scen["letter_of_school"].get(t["school"], ""),
        "justification": justification,
        "education": prof["education"],
        "education_ord": prof["education_ord"],
        "ideology_01": prof["ideology_01"],
        "ideology_label": prof["ideology_label"],
        "religiosity_01": prof["religiosity_01"],
        "religiosity_label": prof["religiosity_label"],
    }


def run_provider(provider, trials, checkpoint_csv=None, checkpoint_every=500):
    """Run every trial for one provider, reporting cache hits separately."""
    cfg = PROVIDER_CFG[provider]
    rows, done, total = [], 0, len(trials)
    hits0 = CACHE_STATS[f"{provider}:hit"]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=cfg["workers"]) as ex:
        futs = {ex.submit(_run_one, provider, t): t for t in trials}
        for fut in as_completed(futs):
            rows.append(fut.result())
            done += 1
            if done % 250 == 0 or done == total:
                el    = time.time() - t0
                hits  = CACHE_STATS[f"{provider}:hit"] - hits0
                rate  = done / max(el, 1e-9)
                eta   = (total - done) / max(rate, 1e-9)
                print(f"    {provider}: {done:,}/{total:,}  "
                      f"[{hits:,} from cache]  "
                      f"({el/60:.1f} min elapsed, ~{eta/60:.1f} min left)")
            if checkpoint_csv and done % checkpoint_every == 0:
                pd.DataFrame(rows).to_csv(checkpoint_csv + ".partial", index=False)
    return pd.DataFrame(rows)


def run_or_load(provider, csv_path, trials):
    out = run_provider(provider, trials, checkpoint_csv=csv_path)
    out.to_csv(csv_path, index=False)
    if os.path.exists(csv_path + ".partial"):
        os.remove(csv_path + ".partial")
    print(f"{provider}: {len(out)} responses; {(~out.valid).sum()} invalid rankings")
    return out


TRIALS = build_trials()
if SMOKE_TEST:
    TRIALS = [t for t in TRIALS if t["agent_id"] < 3]
print(f"Trial grid per provider: {len(TRIALS):,} trials "
      f"({len(TRIALS) * len(PROVIDERS):,} total across {len(PROVIDERS)} providers)")
pd.Series([t["condition"] for t in TRIALS]).value_counts().to_frame("trials")


Trial grid per provider: 4,500 trials (13,500 total across 3 providers)


,trials
C2_School,2000
C3_School_Demo,2000
C1_Baseline,500


In [8]:
frames = []
for provider in PROVIDERS:
    frames.append(run_or_load(provider, os.path.join(RUN_DIR, f"rankings_{provider.lower()}.csv"), TRIALS))
df_all = pd.concat(frames, ignore_index=True)
TRIAL_KEYS = ["provider", "condition", "school", "agent_id", "scenario_id"]
assert not df_all.duplicated(TRIAL_KEYS).any()
assert len(df_all) == len(TRIALS) * len(PROVIDERS)
df_all.to_csv(os.path.join(RUN_DIR, "all_ranked_responses.csv"), index=False)
df = df_all.loc[df_all.valid].copy()
df["rank_tuple"] = df["ranking"].map(lambda s: tuple(json.loads(s)))
quality = df_all.groupby(["provider", "condition", "school"], as_index=False).agg(
    expected=("valid", "size"), valid=("valid", "sum"))
quality["invalid"] = quality.expected - quality.valid
display(quality)


    OpenAI: 250/4,500  [0 from cache]  (0.6 min elapsed, ~10.1 min left)
    OpenAI: 500/4,500  [0 from cache]  (1.2 min elapsed, ~9.3 min left)
    OpenAI: 750/4,500  [0 from cache]  (2.1 min elapsed, ~10.7 min left)
    OpenAI: 1,000/4,500  [1 from cache]  (2.8 min elapsed, ~9.8 min left)
    OpenAI: 1,250/4,500  [2 from cache]  (3.4 min elapsed, ~8.8 min left)
    OpenAI: 1,500/4,500  [2 from cache]  (4.0 min elapsed, ~7.9 min left)
    OpenAI: 1,750/4,500  [2 from cache]  (4.5 min elapsed, ~7.1 min left)
    OpenAI: 2,000/4,500  [2 from cache]  (5.1 min elapsed, ~6.4 min left)
    OpenAI: 2,250/4,500  [2 from cache]  (5.7 min elapsed, ~5.7 min left)
    OpenAI: 2,500/4,500  [2 from cache]  (6.2 min elapsed, ~5.0 min left)
    OpenAI: 2,750/4,500  [2 from cache]  (7.3 min elapsed, ~4.6 min left)
    OpenAI: 3,000/4,500  [2 from cache]  (8.0 min elapsed, ~4.0 min left)
    OpenAI: 3,250/4,500  [2 from cache]  (8.6 min elapsed, ~3.3 min left)
    OpenAI: 3,500/4,500  [2 from cache]  (

,provider,condition,school,expected,valid,invalid
0,Gemini,C1_Baseline,Baseline,500,500,0
1,Gemini,C2_School,Deontology,500,500,0
2,Gemini,C2_School,Teleology,500,500,0
3,Gemini,C2_School,Utilitarianism,500,498,2
4,Gemini,C2_School,VirtueEthics,500,500,0
5,Gemini,C3_School_Demo,Deontology,500,500,0
6,Gemini,C3_School_Demo,Teleology,500,500,0
7,Gemini,C3_School_Demo,Utilitarianism,500,500,0
8,Gemini,C3_School_Demo,VirtueEthics,500,500,0
9,OpenAI,C1_Baseline,Baseline,500,500,0


## 6. Exact Kemeny distance and majority analysis

All calculations operate on rankings of the same four options within a dilemma. Consensus ties are exported in full, without arbitrary tie breaking. A strict majority edge exists only when its count exceeds the reverse count. Cycle detection enumerates simple cycles of lengths three and four.


In [9]:
PERMUTATIONS = list(itertools.permutations(CHOICES))
PAIRS = list(itertools.combinations(CHOICES, 2))

def kemeny_distance(a, b):
    if len(a) != 4 or len(b) != 4 or set(a) != set(CHOICES) or set(b) != set(CHOICES):
        raise ValueError("Rankings must be permutations of A-D")
    pa, pb = {x:i for i,x in enumerate(a)}, {x:i for i,x in enumerate(b)}
    return sum((pa[x] < pa[y]) != (pb[x] < pb[y]) for x,y in PAIRS)

def analyze_profile(rankings):
    rankings = list(rankings)
    if not rankings:
        raise ValueError("Empty electorate")
    counts = Counter(tuple(r) for r in rankings)
    for r in counts:
        kemeny_distance(r, r)
    scores = [sum(n*kemeny_distance(p,r) for r,n in counts.items()) for p in PERMUTATIONS]
    minimum = min(scores)
    optima = [p for p,s in zip(PERMUTATIONS,scores) if s == minimum]
    wins = {a:{b:0 for b in CHOICES} for a in CHOICES}
    for rank,n in counts.items():
        pos = {x:i for i,x in enumerate(rank)}
        for a,b in itertools.permutations(CHOICES,2):
            wins[a][b] += n * (pos[a] < pos[b])
    winner = next((a for a in CHOICES if all(wins[a][b] > wins[b][a] for b in CHOICES if b != a)), None)
    ties = [(a,b) for a,b in PAIRS if wins[a][b] == wins[b][a]]
    cycles = set()
    for length in (3,4):
        for p in itertools.permutations(CHOICES,length):
            if all(wins[p[i]][p[(i+1)%length]] > wins[p[(i+1)%length]][p[i]] for i in range(length)):
                cycles.add(min(p[i:]+p[:i] for i in range(length)))
    return dict(n_voters=len(rankings), kemeny_total=minimum,
                kemeny_per_voter=minimum/len(rankings), kemeny_normalized=minimum/(6*len(rankings)),
                consensus_rankings=json.dumps(optima), n_optima=len(optima),
                condorcet_winner=winner, has_winner=winner is not None,
                tied_pairs=json.dumps(ties), n_tied_pairs=len(ties),
                majority_cycles=json.dumps(sorted(cycles)), has_cycle=bool(cycles),
                pairwise_votes=json.dumps(wins))

# Analytic cases: identical, reversed, distance-two example, a 2-2 tie,
# strict winner, and a three-voter majority cycle.
assert kemeny_distance("ABCD", "ABCD") == 0
assert kemeny_distance("ABCD", "DCBA") == 6
assert kemeny_distance("ABCD", "CABD") == 2
unanimous = analyze_profile(["ABCD"]*4)
assert unanimous["condorcet_winner"] == "A" and unanimous["kemeny_total"] == 0
split = analyze_profile(["ABCD"]*2 + ["DCBA"]*2)
assert not split["has_winner"] and not split["has_cycle"] and split["n_optima"] == 24
cycle = analyze_profile(["ABCD", "BCAD", "CABD"])
assert cycle["has_cycle"] and not cycle["has_winner"]
assert cycle["kemeny_total"] == 4 and cycle["n_optima"] == 3
for a,b in itertools.product(PERMUTATIONS, repeat=2):
    assert kemeny_distance(a,b) == kemeny_distance(b,a)
print("Exact ranking and majority checks passed.")


Exact ranking and majority checks passed.


## 7. Four-school panels and dilemma summaries


In [10]:
PANEL_KEYS = ["provider", "condition", "agent_id", "scenario_id"]
panel_rows, distance_rows, coverage_rows, pool_rows, vote_rows = [], [], [], [], []
ethical = df_all[df_all.condition != "C1_Baseline"]
for key, raw_group in ethical.groupby(PANEL_KEYS, sort=True):
    meta = dict(zip(PANEL_KEYS,key))
    group = df[(df[PANEL_KEYS] == pd.Series(meta)).all(axis=1)]
    complete = len(group) == 4 and set(group.school) == set(SCHOOLS)
    coverage_rows.append({**meta,"complete":complete,"valid_schools":len(group)})
    if not complete:
        continue
    ranks = dict(zip(group.school,group.rank_tuple))
    distances = []
    for a,b in itertools.combinations(SCHOOLS,2):
        distance = kemeny_distance(ranks[a],ranks[b])
        distances.append(distance)
        distance_rows.append({**meta,"school_1":a,"school_2":b,"distance":distance,"normalized_distance":distance/6})
    result = analyze_profile(ranks.values())
    panel_rows.append({**meta,**result,"mean_school_distance":np.mean(distances)})
    for school,rank in ranks.items():
        pool_rows.append({**meta,"school":school,"rank_tuple":rank})
    votes = json.loads(result["pairwise_votes"])
    for a,b in PAIRS:
        vote_rows.append({**meta,"option_a":a,"option_b":b,"votes_a":votes[a][b],"votes_b":votes[b][a],
                          "margin_a":votes[a][b]-votes[b][a]})
coverage = pd.DataFrame(coverage_rows)
panels = pd.DataFrame(panel_rows)
distances = pd.DataFrame(distance_rows)
pairwise_votes = pd.DataFrame(vote_rows)
if panels.empty:
    raise RuntimeError("No complete four-school panels. Inspect all_ranked_responses.csv and repair provider failures.")
GROUP_KEYS = ["provider","condition","scenario_id"]
summary = panels.groupby(GROUP_KEYS,as_index=False).agg(
    complete_panels=("agent_id","size"), mean_kemeny_distance=("mean_school_distance","mean"),
    sd_kemeny_distance=("mean_school_distance","std"), mean_consensus_cost=("kemeny_per_voter","mean"),
    winner_panels=("has_winner","sum"), winner_rate=("has_winner","mean"),
    cycle_rate=("has_cycle","mean"), mean_tied_pairs=("n_tied_pairs","mean"),
    mean_n_optima=("n_optima","mean"))
expected = coverage.groupby(GROUP_KEYS,as_index=False).agg(expected_panels=("complete","size"))
summary = expected.merge(summary,on=GROUP_KEYS,how="left")
summary["complete_panels"] = summary.complete_panels.fillna(0).astype(int)
summary["excluded_panels"] = summary.expected_panels-summary.complete_panels
summary["normalized_mean_distance"] = summary.mean_kemeny_distance/6
school_pairs = distances.groupby(["provider","condition","school_1","school_2"],as_index=False).agg(
    n_panels=("distance","size"), mean_distance=("distance","mean"), sd_distance=("distance","std"))
winner_counts = panels.assign(winner=panels.condorcet_winner.fillna("None")).groupby(
    GROUP_KEYS+["winner"],as_index=False).size()
pooled_rows = []
for key,g in pd.DataFrame(pool_rows).groupby(GROUP_KEYS,sort=True):
    pooled_rows.append({**dict(zip(GROUP_KEYS,key)),**analyze_profile(g.rank_tuple)})
pooled = pd.DataFrame(pooled_rows)
pooled_frequency = pooled.groupby(["provider","condition"],as_index=False).agg(
    analyzed_dilemmas=("scenario_id","size"), winner_dilemmas=("has_winner","sum"),
    winner_rate=("has_winner","mean"), cycle_dilemmas=("has_cycle","sum"))
display(summary)
display(pooled_frequency)


,provider,condition,scenario_id,expected_panels,complete_panels,mean_kemeny_distance,sd_kemeny_distance,mean_consensus_cost,winner_panels,winner_rate,cycle_rate,mean_tied_pairs,mean_n_optima,excluded_panels,normalized_mean_distance
0,Gemini,C2_School,1,50,50,2.000000,0.000000,1.500000,0,0.000000,0.0,3.000000,6.000000,0,0.333333
1,Gemini,C2_School,2,50,48,1.607639,0.258404,0.848958,48,1.000000,0.0,0.270833,1.270833,2,0.267940
2,Gemini,C2_School,3,50,50,1.083333,0.261363,0.575000,50,1.000000,0.0,0.200000,1.200000,0,0.180556
3,Gemini,C2_School,4,50,50,2.400000,0.416497,1.200000,50,1.000000,0.0,0.000000,1.000000,0,0.400000
4,Gemini,C2_School,5,50,50,3.043333,0.073848,1.565000,39,0.780000,0.0,0.260000,1.260000,0,0.507222
5,Gemini,C2_School,6,50,50,3.190000,0.095298,1.785000,8,0.160000,0.0,1.140000,2.380000,0,0.531667
6,Gemini,C2_School,7,50,50,2.806667,0.226779,1.570000,50,1.000000,0.0,1.000000,2.000000,0,0.467778
7,Gemini,C2_School,8,50,50,3.016667,0.127108,1.545000,39,0.780000,0.0,0.220000,1.220000,0,0.502778
8,Gemini,C2_School,9,50,50,3.023333,0.058418,1.535000,43,0.860000,0.0,0.140000,1.140000,0,0.503889
9,Gemini,C2_School,10,50,50,2.580000,0.301621,1.450000,4,0.080000,0.0,0.960000,1.960000,0,0.430000


,provider,condition,analyzed_dilemmas,winner_dilemmas,winner_rate,cycle_dilemmas
0,Gemini,C2_School,10,9,0.9,0
1,Gemini,C3_School_Demo,10,10,1.0,0
2,OpenAI,C2_School,10,9,0.9,0
3,OpenAI,C3_School_Demo,10,10,1.0,0
4,Together,C2_School,10,9,0.9,0
5,Together,C3_School_Demo,10,10,1.0,0


## 8. Matched baseline comparisons and demographic shifts

Distances compare full rankings for the same identity and dilemma. C1 is compared separately with each school in C2 and C3. C2–C3 shifts include full-ranking changes and top-option changes. Missing pairs are excluded and their denominators are reported.


In [11]:
MATCH_KEYS = ["provider","agent_id","scenario_id"]
base = df[df.condition == "C1_Baseline"][MATCH_KEYS+["rank_tuple"]]
baseline_rows=[]
for condition in ["C2_School","C3_School_Demo"]:
    for school in SCHOOLS:
        other = df[(df.condition == condition)&(df.school == school)]
        joined = base.merge(other,on=MATCH_KEYS,suffixes=("_base","_school"),validate="one_to_one")
        for _,r in joined.iterrows():
            baseline_rows.append({**{k:r[k] for k in MATCH_KEYS},"condition":condition,"school":school,
                                  "distance":kemeny_distance(r.rank_tuple_base,r.rank_tuple_school)})
baseline_distances=pd.DataFrame(baseline_rows,columns=MATCH_KEYS+["condition","school","distance"])
baseline_summary=baseline_distances.groupby(["provider","condition","school"],as_index=False).agg(
    matched_pairs=("distance","size"),mean_distance=("distance","mean"))
c2=df[df.condition=="C2_School"]
c3=df[df.condition=="C3_School_Demo"]
matched=c2.merge(c3,on=MATCH_KEYS+["school"],suffixes=("_c2","_c3"),validate="one_to_one")
shift_rows=[]
for _,r in matched.iterrows():
    d=kemeny_distance(r.rank_tuple_c2,r.rank_tuple_c3)
    shift_rows.append({**{k:r[k] for k in MATCH_KEYS+["school"]},"distance":d,
                      "ranking_changed":d>0,"top_changed":r.rank_tuple_c2[0]!=r.rank_tuple_c3[0]})
shifts=pd.DataFrame(shift_rows,columns=MATCH_KEYS+["school","distance","ranking_changed","top_changed"])
shift_summary=shifts.groupby(["provider","school"],as_index=False).agg(
    matched_pairs=("distance","size"),mean_distance=("distance","mean"),
    ranking_change_rate=("ranking_changed","mean"),top_change_rate=("top_changed","mean"))
expected_pairs = len({t["agent_id"] for t in TRIALS})*len(SCENARIOS)
for table in (baseline_summary,shift_summary):
    table["expected_pairs"]=expected_pairs
    table["excluded_pairs"]=expected_pairs-table.matched_pairs
panel_shifts=panels[panels.condition=="C2_School"].merge(
    panels[panels.condition=="C3_School_Demo"],on=MATCH_KEYS,suffixes=("_c2","_c3"),validate="one_to_one")
panel_shifts["distance_change"] = panel_shifts.mean_school_distance_c3-panel_shifts.mean_school_distance_c2
panel_shifts["winner_status_changed"] = panel_shifts.has_winner_c2 != panel_shifts.has_winner_c3
display(baseline_summary)
display(shift_summary)


,provider,condition,school,matched_pairs,mean_distance,expected_pairs,excluded_pairs
0,Gemini,C2_School,Deontology,500,3.808000,500,0
1,Gemini,C2_School,Teleology,500,1.632000,500,0
2,Gemini,C2_School,Utilitarianism,498,1.771084,500,2
3,Gemini,C2_School,VirtueEthics,500,2.432000,500,0
4,Gemini,C3_School_Demo,Deontology,500,3.866000,500,0
5,Gemini,C3_School_Demo,Teleology,500,1.580000,500,0
6,Gemini,C3_School_Demo,Utilitarianism,500,1.790000,500,0
7,Gemini,C3_School_Demo,VirtueEthics,500,2.350000,500,0
8,OpenAI,C2_School,Deontology,500,2.244000,500,0
9,OpenAI,C2_School,Teleology,500,0.476000,500,0


,provider,school,matched_pairs,mean_distance,ranking_change_rate,top_change_rate,expected_pairs,excluded_pairs
0,Gemini,Deontology,500,0.290000,0.254000,0.140000,500,0
1,Gemini,Teleology,500,0.656000,0.466000,0.154000,500,0
2,Gemini,Utilitarianism,498,0.751004,0.323293,0.236948,500,2
3,Gemini,VirtueEthics,500,0.330000,0.310000,0.226000,500,0
4,OpenAI,Deontology,499,0.104208,0.104208,0.094188,500,1
5,OpenAI,Teleology,500,0.364000,0.288000,0.108000,500,0
6,OpenAI,Utilitarianism,500,0.570000,0.386000,0.286000,500,0
7,OpenAI,VirtueEthics,500,0.472000,0.348000,0.144000,500,0
8,Together,Deontology,492,0.725610,0.477642,0.321138,500,8
9,Together,Teleology,482,0.842324,0.562241,0.342324,500,18


## 9. Figures, tables, and Word report

The report includes actual run coverage, school-pair distances, dilemma-level Kemeny and Condorcet results, all pooled optimal rankings, pairwise majority counts, and matched comparisons. Full panel-level results and raw responses are exported for audit. Percentages use valid complete panels as denominators; pooled elections are a separate estimand.


In [ ]:
from pathlib import Path
from docx.shared import Inches, Pt
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

fig_paths=[]
for provider in PROVIDERS:
    sub=summary[summary.provider==provider]
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for condition,g in sub.groupby("condition"):
        axes[0].plot(g.scenario_id,g.mean_kemeny_distance,marker="o",label=CONDITION_LABEL[condition])
        axes[1].plot(g.scenario_id,g.winner_rate,marker="o",label=CONDITION_LABEL[condition])
    axes[0].set(title="Mean school-pair Kemeny distance",ylim=(0,6),xlabel="Dilemma",ylabel="Discordant pairs")
    axes[1].set(title="Four-school panels with a Condorcet winner",ylim=(0,1.05),xlabel="Dilemma",ylabel="Proportion")
    axes[0].legend(fontsize=8)
    fig.suptitle(PROVIDER_LABEL[provider]);fig.tight_layout()
    path=os.path.join(FIG_DIR,f"{provider.lower()}_social_choice.png")
    fig.savefig(path,dpi=180);plt.close(fig);fig_paths.append(path)

tables={"quality":quality,"option_key":key_tbl,"profiles":profiles_df,
        "panel_coverage":coverage,"panels":panels,"school_distances":distances,
        "school_pair_summary":school_pairs,"dilemma_summary":summary,
        "pairwise_votes":pairwise_votes,"winner_counts":winner_counts,
        "pooled_elections":pooled,"pooled_frequency":pooled_frequency,
        "baseline_pairs":baseline_distances,"baseline_summary":baseline_summary,
        "demographic_pairs":shifts,"demographic_summary":shift_summary,"panel_shifts":panel_shifts}
with pd.ExcelWriter(RESULTS_XLSX,engine="openpyxl") as writer:
    for name,table in tables.items():
        table.to_excel(writer,sheet_name=name,index=False)
        table.to_csv(os.path.join(RUN_DIR,f"{name}.csv"),index=False)
    # Raw text remains in CSV, avoiding Excel's 32,767-character cell limit.
    for ws in writer.book.worksheets:
        ws.freeze_panes="A2";ws.auto_filter.ref=ws.dimensions

manifest={"models":PROVIDER_MODEL,"seed":RANDOM_SEED,"n_agents_sampled":N_AGENTS,
          "smoke_test":SMOKE_TEST,"trials_per_provider":len(TRIALS),"max_tokens":MAX_TOKENS,
          "temperature_by_identity":AGENT_TEMPERATURE,"demographic_source":DEMO_PARQUET,
          "output_contract":OUTPUT_CONTRACT,"personas":PERSONA_PROMPTS,
          "input_sha256":{p:hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in [DOCX_PATH,ANSWER_KEY]},
          "created_utc":pd.Timestamp.now(tz="UTC").isoformat()}
Path(RUN_DIR,"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

report=Document()
normal=report.styles["Normal"];normal.font.name="Calibri";normal.font.size=Pt(10)
section=report.sections[0]
section.top_margin=section.bottom_margin=Inches(.7)
section.left_margin=section.right_margin=Inches(.7)
report.add_heading("Ethical School Ranking Analysis",0)
report.add_paragraph("Kemeny Distance and Condorcet Winner Results")
report.add_paragraph(f"Run: {'SMOKE TEST — preliminary subset' if SMOKE_TEST else 'full experiment'}. "
    f"{len(df_all):,} trials; {int(df_all.valid.sum()):,} valid full rankings; "
    f"{len(panels):,} complete four-school panels; {int((~coverage.complete).sum()):,} excluded panels.")
report.add_heading("Experimental setup",1)
report.add_paragraph(f"The experiment uses {len(SCENARIOS)} randomized dilemmas, "
    f"{len({t['agent_id'] for t in TRIALS})} active matched identities, and three conditions: baseline, "
    "ethical school, and ethical school with demographics. Personas, US profile sampling, seed, "
    "temperature assignment, and model identifiers follow the original experiment. Every agent ranks "
    "all four presented options best to worst. Rankings are mapped to canonical ethical labels only after collection.")
report.add_paragraph("Models: "+"; ".join(f"{p}: {m}" for p,m in PROVIDER_MODEL.items()))
report.add_paragraph("Canonical option labels: A = Utilitarianism; B = Deontology; C = Virtue Ethics; "
    "D = Teleology. These are answer-key labels, not claims about moral correctness. "
    "Displayed consensus rankings and winners below use these canonical labels; the workbook option key maps them back to each dilemma's presented letters.")

def report_table(frame,columns=None):
    frame=frame if columns is None else frame[columns]
    table=report.add_table(rows=1,cols=len(frame.columns));table.style="Light Shading Accent 1"
    for cell,name in zip(table.rows[0].cells,frame.columns):cell.text=str(name).replace("_"," ")
    repeat=OxmlElement("w:tblHeader");table.rows[0]._tr.get_or_add_trPr().append(repeat)
    for row in frame.itertuples(index=False,name=None):
        for cell,value in zip(table.add_row().cells,row):
            cell.text=("—" if pd.isna(value) else f"{value:.3f}" if isinstance(value,(float,np.floating)) else str(value))
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:run.font.size=Pt(8)
    report.add_paragraph()

report.add_heading("Analysis definitions and coverage",1)
report.add_paragraph("Kemeny distance is the number of discordant unordered option pairs, from 0 to 6. "
    "Each complete panel contains one ranking per school for a matched identity and dilemma. "
    "The reported panel distance averages the six school-pair distances. Exact Kemeny consensus "
    "enumerates all 24 permutations and minimizes total distance to the voters; all tied optima are retained. "
    "Consensus cost per voter differs from average pairwise disagreement.")
report.add_paragraph("A strict Condorcet winner beats each other option head to head. Tied contests do not count "
    "as wins. Majority cycles and tied pairs are recorded separately: absence of a winner does not establish a cycle. "
    "Cycles may also occur among losing alternatives. Pooled elections combine complete four-school panels "
    "within a dilemma, preserving equal school representation. They answer a different question from individual panels.")
report_table(quality)
for provider in PROVIDERS:
    report.add_heading(PROVIDER_LABEL[provider],1)
    report.add_picture(fig_paths[PROVIDERS.index(provider)],width=Inches(6.8))
    for condition in ["C2_School","C3_School_Demo"]:
        report.add_heading(CONDITION_LABEL[condition],2)
        sub=summary[(summary.provider==provider)&(summary.condition==condition)]
        report.add_paragraph("Dilemma results: distance is the mean across complete panels; winner and cycle rates "
            "are proportions of complete panels. Excluded counts include panels with any invalid school ranking.")
        report_table(sub,["scenario_id","complete_panels","excluded_panels","mean_kemeny_distance","winner_rate","cycle_rate","mean_tied_pairs"])
        pairs=school_pairs[(school_pairs.provider==provider)&(school_pairs.condition==condition)]
        report.add_paragraph("School-pair distances across observed complete panels")
        report_table(pairs,["school_1","school_2","n_panels","mean_distance"])
        freq=pooled_frequency[(pooled_frequency.provider==provider)&(pooled_frequency.condition==condition)]
        if len(freq):
            f=freq.iloc[0]
            report.add_paragraph(f"The pooled electorate has a strict Condorcet winner in {int(f.winner_dilemmas)} "
                f"of {int(f.analyzed_dilemmas)} analyzed dilemmas ({f.winner_rate:.1%}).")
        for _,r in pooled[(pooled.provider==provider)&(pooled.condition==condition)].iterrows():
            title=next(s["title"] for s in SCENARIOS if s["id"]==r.scenario_id)
            report.add_heading(f"Dilemma {r.scenario_id}: {title}",3)
            winner=r.condorcet_winner if r.has_winner else "None"
            orders="; ".join(" > ".join(x) for x in json.loads(r.consensus_rankings))
            report.add_paragraph(f"Pooled voters: {r.n_voters}. Condorcet winner: {winner}. "
                f"Tied pairs: {r.n_tied_pairs}. Majority cycle present: {r.has_cycle}. "
                f"Minimum total Kemeny cost: {r.kemeny_total}; cost per voter: {r.kemeny_per_voter:.3f}. "
                f"All {r.n_optima} optimal ranking(s): {orders}.")
            votes=json.loads(r.pairwise_votes)
            report_table(pd.DataFrame([{"Pair":f"{a} vs {b}","Votes for first":votes[a][b],
                                       "Votes for second":votes[b][a]} for a,b in PAIRS]))
report.add_heading("Matched baseline comparisons",1)
report_table(baseline_summary)
report.add_heading("Matched demographic shifts",1)
report_table(shift_summary)
report.add_heading("Interpretation and limitations",1)
report.add_paragraph("Lower Kemeny distances indicate more similar orderings, not greater ethical correctness. "
    "Condorcet winners indicate pairwise majority support within the specified electorate. "
    "The same sampled identities recur across dilemmas and conditions, so these summaries are descriptive; "
    "no independence-based significance claims are made. Invalid responses may create selection bias; "
    "coverage tables report their extent. Pooled results and matched four-school panel results should not be conflated. "
    "Model outputs can vary despite identical temperature settings, and these simulated identities do not establish human population preferences.")
report.add_paragraph("The Excel workbook and CSVs contain all panel consensus ties, pairwise vote counts, "
    "cycle lists, matched distances, and coverage records. Raw replies are in all_ranked_responses.csv; "
    "manifest.json records configuration and input hashes.")
report.save(REPORT_DOCX)
print("Word report:",REPORT_DOCX)
print("Excel tables:",RESULTS_XLSX)
print("Audit data and figures:",RUN_DIR)
